# 02 — Model Version Ladder

Side-by-side comparison of every model version so far. Re-run after training a new version to see where it lands.

**Reads only:** `artifacts/metrics/*.json` (written by each model's training script).

**Test set is held out** — only train / val numbers are shown here. The test set comparison happens once at the end of the project.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from ufc_pred.paths import METRICS, MODELS

pd.set_option('display.precision', 4)

## Load all version runs

In [ ]:
runs = []
for p in sorted(METRICS.glob('*.json')):
    with open(p) as f:
        payload = json.load(f)
    # Skip non-model JSONs (e.g. bet_eval.json, which lives in the same dir).
    if 'version' not in payload or 'metrics' not in payload:
        continue
    runs.append(payload)

print(f'Found {len(runs)} version run(s):')
for r in runs:
    print(f"  - {r['version']}  trained {r['trained_at']}")

## Comparison table — log loss is the primary metric

In [ ]:
rows = []
for r in runs:
    m = r['metrics']
    cols = ['n', 'log_loss', 'brier', 'ece', 'accuracy_argmax']
    # 'train' metrics are missing on the retrained _full2000 artifacts.
    if 'train' in m:
        rows.append({'version': r['version'], 'split': 'train', **{k: m['train'][k] for k in cols}})
    if 'val' in m:
        rows.append({'version': r['version'], 'split': 'val',   **{k: m['val'][k]   for k in cols}})

# Market baseline (val) — taken from whichever run has it; should be identical across runs.
market = next((r['metrics']['val_market'] for r in runs if 'val_market' in r['metrics']), None)
if market is not None:
    rows.append({'version': 'market', 'split': 'val', **{k: market[k] for k in ['n', 'log_loss', 'brier', 'ece', 'accuracy_argmax']}})
rows.append({'version': 'v0 coinflip', 'split': 'val', 'n': market['n'] if market else None, 'log_loss': 0.6931, 'brier': 0.25, 'ece': np.nan, 'accuracy_argmax': 0.5})

ladder = pd.DataFrame(rows)[['version', 'split', 'n', 'log_loss', 'brier', 'ece', 'accuracy_argmax']]
ladder

### Just the val numbers, side-by-side

In [ ]:
val_only = ladder[ladder.split == 'val'].drop(columns='split').sort_values('log_loss')
val_only.style.format({'log_loss': '{:.4f}', 'brier': '{:.4f}', 'ece': '{:.4f}', 'accuracy_argmax': '{:.3f}'}).background_gradient(subset=['log_loss', 'brier'], cmap='RdYlGn_r')

## Calibration / reliability curves

Re-compute probabilities on the validation set for each saved model and plot empirical-vs-predicted in 10 buckets. Closer to the diagonal = better calibration.

In [ ]:
from ufc_pred.ingest.kaggle_mdabbert import HISTORY_PARQUET
from ufc_pred.utils.time_splits import split
from ufc_pred.features.static_v1 import prepare
from ufc_pred.features.skill_v3_pipeline import OUTPUT as SKILL_PARQUET
from ufc_pred.features.skill_v3_1_pipeline import OUTPUT as SKILL_V3_1_PARQUET
from ufc_pred.features.derived_v2 import compute as add_derived_v2
from ufc_pred.backtest.metrics import market_no_vig_prob_red

fights = pd.read_parquet(HISTORY_PARQUET)
fights = fights[fights.Winner.isin(['Red', 'Blue'])].copy()
fights['date'] = pd.to_datetime(fights['date'])

# Join v3 skill features (named for v3 / v3.1 / v3.2 / v3_full2000 / v3.1_full2000).
_skill = pd.read_parquet(SKILL_PARQUET)
_skill['date'] = pd.to_datetime(_skill['date'])
fights = fights.merge(
    _skill[['date', 'R_fighter', 'B_fighter', 'skill_diff_mean', 'skill_diff_std']],
    on=['date', 'R_fighter', 'B_fighter'], how='left', validate='many_to_one',
)

# Also join v3 + v3.1 skill features under suffixed names for v3.3 / v7 / v3.3_full2000.
_sk_v3 = pd.read_parquet(SKILL_PARQUET).rename(columns={
    'skill_diff_mean': 'skill_diff_mean_v3', 'skill_diff_std': 'skill_diff_std_v3'})
_sk_v3['date'] = pd.to_datetime(_sk_v3['date'])
_sk_v3_1 = pd.read_parquet(SKILL_V3_1_PARQUET).rename(columns={
    'skill_diff_mean': 'skill_diff_mean_v3_1', 'skill_diff_std': 'skill_diff_std_v3_1'})
_sk_v3_1['date'] = pd.to_datetime(_sk_v3_1['date'])
fights = fights.merge(_sk_v3[['date','R_fighter','B_fighter','skill_diff_mean_v3','skill_diff_std_v3']],
                     on=['date','R_fighter','B_fighter'], how='left', validate='many_to_one')
fights = fights.merge(_sk_v3_1[['date','R_fighter','B_fighter','skill_diff_mean_v3_1','skill_diff_std_v3_1']],
                     on=['date','R_fighter','B_fighter'], how='left', validate='many_to_one')

# v2 derived features (layoff, activity, ratios) — older models ignore via reindex.
fights = add_derived_v2(fights)

splits = split(fights)

# Two prepared views of val: one one-hot (logreg), one raw cats (CatBoost / similar).
X_val_oh,  y_val, d_val, _ = prepare(splits.val, augment_symmetry=False, one_hot=True)
X_val_raw, _,    _,    _ = prepare(splits.val, augment_symmetry=False, one_hot=False)

def _predict(payload, X_oh, X_raw):
    cols = payload['columns']
    one_hot = payload.get('one_hot', True)
    Xv = (X_oh if one_hot else X_raw).reindex(columns=cols, fill_value=0)
    if not one_hot:
        for c in payload.get('cat_features', []):
            if c in Xv.columns:
                Xv[c] = Xv[c].fillna('__missing__').astype(str)
    # Ensemble payload (v7, v7.1).
    if 'models' in payload:
        preds = [m.predict_proba(Xv)[:, 1] for m in payload['models']]
        return np.mean(np.vstack(preds), axis=0)
    # Standard single-model payload.
    if 'model' in payload:
        return payload['model'].predict_proba(Xv)[:, 1]
    # v1_2 isotonic: load the underlying ranker from disk, then apply isotonic.
    if 'ranker_path' in payload and 'isotonic' in payload:
        ranker = joblib.load(payload['ranker_path'])['model']
        raw = ranker.predict_proba(Xv)[:, 1]
        return payload['isotonic'].transform(raw)
    raise KeyError(f'unrecognized payload keys: {list(payload.keys())}')

model_curves = {}
for r in runs:
    payload = joblib.load(r['model_path'])
    model_curves[r['version']] = _predict(payload, X_val_oh, X_val_raw)

# Market on the same set
sub = splits.val.dropna(subset=['R_odds', 'B_odds']).copy()
y_mkt = (sub.Winner == 'Red').astype(int).to_numpy()
p_mkt = market_no_vig_prob_red(sub)

In [ ]:
def reliability(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins - 1)
    pred = np.array([p[idx == b].mean() if (idx == b).any() else np.nan for b in range(n_bins)])
    actual = np.array([y[idx == b].mean() if (idx == b).any() else np.nan for b in range(n_bins)])
    return pred, actual

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect')
for name, p in model_curves.items():
    pred, actual = reliability(y_val, p)
    ax.plot(pred, actual, 'o-', label=name)
pred, actual = reliability(y_mkt, p_mkt)
ax.plot(pred, actual, 's-', label='market (no-vig)', color='black')
ax.set_xlabel('Predicted P(Red wins)'); ax.set_ylabel('Empirical P(Red wins)')
ax.set_title('Reliability — validation set'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Probability distribution per model

Are predictions concentrated near 0.5 (cautious model) or spread out (confident)? A model that never strays from 0.5 won't generate any bets — even if calibrated.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 21)
for name, p in model_curves.items():
    ax.hist(p, bins=bins, alpha=0.45, label=name)
ax.hist(p_mkt, bins=bins, alpha=0.45, label='market', color='black', histtype='step', linewidth=2)
ax.set_xlabel('Predicted P(Red wins)'); ax.set_ylabel('count')
ax.set_title('Predicted probability distribution — validation set'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Time-decay analysis — does the model get worse as we move further from training?

Models can stale as fighters age, get injured, change camps, or the metagame shifts. We test this by bucketing the **validation set by month** and computing log loss per bucket. If early-val months (close to the training cutoff) score better than late-val months, the model has drift and we'll want to retrain more often (or use walk-forward).

The training cutoff is **2022-12-31**, so val month 1 = January 2023 (~1 month gap), val month 12 = December 2023 (~12 month gap).

In [ ]:
from ufc_pred.backtest.metrics import evaluate
from ufc_pred.utils.time_splits import TRAIN_END

# Align dates to val predictions (already aligned by row order from prepare()).
val_df = pd.DataFrame({'date': pd.to_datetime(d_val), 'y': y_val})
for name, p in model_curves.items():
    val_df[name] = p

# Bucket by year-month
val_df['month'] = val_df['date'].dt.to_period('M')
buckets = sorted(val_df['month'].unique())

rows = []
for m in buckets:
    sub = val_df[val_df['month'] == m]
    if len(sub) < 10:
        continue
    gap_days = (sub['date'].mean() - TRAIN_END).days
    row = {'month': str(m), 'n': len(sub), 'gap_days': gap_days}
    for name in model_curves:
        row[name] = evaluate(sub['y'].to_numpy(), sub[name].to_numpy())['log_loss']
    rows.append(row)

decay_df = pd.DataFrame(rows)
decay_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in model_curves:
    ax.plot(decay_df['gap_days'], decay_df[name], 'o-', label=name)

# np.polyfit returns [slope, intercept] (highest power first) for deg=1.
for name in model_curves:
    slope, intercept = np.polyfit(decay_df['gap_days'], decay_df[name], 1)
    xs = np.array([decay_df['gap_days'].min(), decay_df['gap_days'].max()])
    ax.plot(xs, intercept + slope * xs, '--', alpha=0.5,
            label=f'{name} trend ({slope * 30:+.4f}/month)')

ax.axhline(0.6931, color='gray', linestyle=':', alpha=0.5, label='coin flip')
ax.set_xlabel('Days since training cutoff (2022-12-31)')
ax.set_ylabel('Log loss (lower is better)')
ax.set_title('Model performance vs time gap — validation set')
ax.set_ylim(0.4, 0.75)
ax.legend(loc='best', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nSlope interpretation: positive → model degrades as gap grows; negative → improves.')
print('Caveat: only 504 val fights across 12 months → noisy. Treat as a hint, not proof.')

## v3 — Bayesian skill features

v3 adds two columns from [data/processed/skill_features_v3.parquet](../data/processed/skill_features_v3.parquet):
`skill_diff_mean` and `skill_diff_std`, the posterior mean / std of `(skill[R] − skill[B])` from a hierarchical
Bradley-Terry model fit walk-forward (monthly cadence; posterior for fight at date X uses only fights with date < X).

The plots below dig into the features themselves — not v3's predictions (those are already in the reliability /
distribution / time-decay charts above). Goal: do the skill features actually carry signal, and how does v3 differ
from v1.1 fight-by-fight?

In [ ]:
from ufc_pred.features.skill_v3_pipeline import OUTPUT as SKILL_PARQUET

skill = pd.read_parquet(SKILL_PARQUET)
skill['date'] = pd.to_datetime(skill['date'])
print('skill_features_v3 rows:', len(skill))
print('NaN rows (early months, < min_train_fights):', skill['skill_diff_mean'].isna().sum())
display(skill.describe())

# `fights` (from the reliability cell above) already has the skill columns joined.
val_skill = splits.val.copy()
val_skill['y'] = (val_skill['Winner'] == 'Red').astype(int)
print(f"\nval skill coverage: {val_skill['skill_diff_mean'].notna().mean():.3f}")

### Feature distributions

`skill_diff_mean` should be roughly symmetric around 0 (the model is symmetric in R/B). `skill_diff_std` is the
posterior uncertainty — high for fights involving small-sample fighters or recent debuts, low for established matchups.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(val_skill['skill_diff_mean'].dropna(), bins=30, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('skill_diff_mean (Red − Blue)')
axes[0].set_ylabel('count')
axes[0].set_title('Posterior mean skill differential — val set')
axes[0].grid(alpha=0.3)

axes[1].hist(val_skill['skill_diff_std'].dropna(), bins=30, alpha=0.7, color='darkorange', edgecolor='white')
axes[1].set_xlabel('skill_diff_std')
axes[1].set_ylabel('count')
axes[1].set_title('Posterior uncertainty — val set')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(val_skill[['skill_diff_mean', 'skill_diff_std']].describe().to_string(float_format=lambda x: f'{x:.3f}'))

### Does `skill_diff_mean` actually predict who wins?

Bucket val fights by skill differential and plot empirical Red win rate per bucket. If the feature carries signal,
the line should slope upward — fights where Red has a higher posterior skill should win more often. Compare to the
diagonal (sigmoid of the differential) — if our curve tracks the sigmoid, the Bayesian model's logit-link assumption
is roughly honored on held-out data.

In [ ]:
vs = val_skill.dropna(subset=['skill_diff_mean']).copy()
# Equal-count buckets so every point is non-trivially populated.
vs['bin'] = pd.qcut(vs['skill_diff_mean'], q=10, duplicates='drop')
bin_stats = vs.groupby('bin', observed=True).agg(
    n=('y', 'size'),
    win_rate=('y', 'mean'),
    skill_mid=('skill_diff_mean', 'mean'),
)

fig, ax = plt.subplots(figsize=(7, 5))
xs = np.linspace(vs['skill_diff_mean'].min(), vs['skill_diff_mean'].max(), 200)
ax.plot(xs, 1 / (1 + np.exp(-xs)), 'k--', alpha=0.5, label='sigmoid(skill_diff)')
ax.errorbar(
    bin_stats['skill_mid'], bin_stats['win_rate'],
    yerr=np.sqrt(bin_stats['win_rate'] * (1 - bin_stats['win_rate']) / bin_stats['n']),
    fmt='o-', color='steelblue', label='empirical (val, 10 quantile bins)',
)
ax.axhline(0.5, color='gray', alpha=0.3); ax.axvline(0, color='gray', alpha=0.3)
ax.set_xlabel('skill_diff_mean'); ax.set_ylabel('P(Red wins)')
ax.set_title('Skill differential vs empirical win rate — val set')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

display(bin_stats)

### v3 vs v1.1 — where do the predictions actually differ?

The big question: is v3 *just* v1.1 with a tiny shift everywhere, or does it move specific fights meaningfully? If
the deltas cluster near 0 with a few notable shifts, the model is using skill features selectively. If everything
shifts uniformly, skill features are mostly redundant with what v1.1 already learned.

In [ ]:
v11 = next((k for k in model_curves if 'v1_1' in k), None)
v3  = next((k for k in model_curves if 'v3'   in k), None)

if v11 and v3:
    p11, p3 = model_curves[v11], model_curves[v3]
    delta = p3 - p11
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(p11, p3, c=val_skill['skill_diff_mean'].fillna(0), cmap='RdBu_r',
                    s=20, alpha=0.7, edgecolor='none', vmin=-0.5, vmax=0.5)
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[0].set_xlabel(f'{v11}  P(Red)'); axes[0].set_ylabel(f'{v3}  P(Red)')
    axes[0].set_title('Per-fight predictions — color = skill_diff_mean')
    axes[0].grid(alpha=0.3)

    axes[1].hist(delta, bins=40, color='purple', alpha=0.7, edgecolor='white')
    axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)
    axes[1].set_xlabel(f'{v3} − {v11}  (Δ P(Red))')
    axes[1].set_ylabel('count')
    axes[1].set_title('Prediction shift introduced by skill features')
    axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()

    print(f'|Δ| mean: {np.abs(delta).mean():.3f}   max: {np.abs(delta).max():.3f}   '
          f'fights shifted by >0.05: {(np.abs(delta) > 0.05).sum()} / {len(delta)}')

    # Top 8 biggest shifts on val
    val_idx = splits.val.reset_index(drop=True)
    top = pd.DataFrame({
        'date': val_idx['date'].dt.date,
        'R': val_idx['R_fighter'], 'B': val_idx['B_fighter'],
        'winner': val_idx['Winner'],
        'p_v1_1': p11, 'p_v3': p3, 'delta': delta,
        'skill_diff_mean': val_skill.reset_index(drop=True)['skill_diff_mean'],
    }).assign(abs_delta=lambda d: d['delta'].abs()).sort_values('abs_delta', ascending=False).head(8)
    print('\nLargest |Δ| fights on val:')
    display(top.drop(columns='abs_delta'))
else:
    print('Need both v1.1 and v3 trained for this comparison. Found:', list(model_curves))

## v2 — Derived no-scrape features (verdict: kill on log loss)

v2 adds 10 columns to v1.1 derived purely from existing data (no scraping):
layoff days, fights-in-last-365d / 730d, career totals, win rate, finish rate. See
[derived_v2.py](../src/ufc_pred/features/derived_v2.py).

**Result:** val log_loss 0.6183 vs v1.1's 0.6190 — a 0.11% relative improvement,
**below the 0.5% keep-or-kill bar**. The Kaggle feature set is effectively saturated;
no free lunch remains in the existing columns. Real signal from here needs new
information (UFCStats scrape → v2.1: defense %, control time, knockdowns).

**Side note worth keeping:** v2's ECE = 0.0286, less than half of v1.1's 0.062
(and better than v3's 0.060). Accuracy dropped (0.643 vs 0.683) — the model
became less confident and better calibrated. Possibly useful to revisit in Phase 7
where calibration is the target rather than log loss.


In [ ]:
v2_run = next((r for r in runs if 'v2' in r['version']), None)
if v2_run is None:
    print('v2 not trained yet')
else:
    payload = joblib.load(v2_run['model_path'])
    model = payload['model']
    feats = payload['columns']
    imp = pd.Series(model.get_feature_importance(), index=feats).sort_values(ascending=False)
    derived_cols = v2_run['derived_features']['columns']
    print('=== v2 derived-feature importance (out of', len(feats), 'features) ===')
    print(imp.loc[[c for c in derived_cols if c in imp.index]].to_string())
    print()
    print('=== top 15 features overall in v2 ===')
    print(imp.head(15).to_string())
    print()
    print('=== rank of each derived feature ===')
    ranks = imp.rank(ascending=False).astype(int)
    print(ranks.loc[[c for c in derived_cols if c in ranks.index]].sort_values().to_string())


## v3.2 — v3 skill + v2 derived (verdict: kill, actively worse than v3 alone)

Combo test: does stacking v3's Bayesian skill features with v2's derived features
give us the best of both — v3's log-loss win plus v2's calibration win?

**No.** Val log_loss 0.6191 — *worse than v3's 0.6119 and identical to v1.1*.
Brier 0.215 (vs v3's 0.211). ECE 0.046 (between v2's 0.029 and v3's 0.060).

The smoking gun is **best_iteration = 213** (vs v2's 642 and typical 800–1500).
CatBoost overfit train ~3× faster than usual, meaning the enlarged feature set
let it memorize the training set before validation peaked.

**Likely mechanism:** v3's Bayesian skill posterior already implicitly encodes
most of what `R_days_since_last_fight` etc. carry — the recency-weighted
likelihood (half-life 4y) means infrequent fighters get wider, less informative
posteriors. Stacking the two creates redundancy that competes for splits without
adding new information.

**Takeaway:** v3 stays the current best. Do not propagate v2's features into v3.1 / v4.
The keep-or-kill rule caught what would otherwise have been a slow accretion of noise.


## Notes & decisions

Use this section to jot what you learned after each version. The numbers above will be obsolete in a week; your notes about *why* a version did what it did will not.

### v1 — logistic regression, no odds
- _your notes here_

### v1.1 — CatBoost, same features
- _your notes here_
### v2 — CatBoost + derived no-scrape features (layoff, activity, ratios)
- Val log_loss 0.6183 vs v1.1 0.6190 → 0.11% relative, **below the 0.5% bar → kill on log loss**.
- Kaggle features are saturated; real signal needs UFCStats scrape (defense %, control time).
- ECE dropped to 0.0286 (less than half v1.1's 0.062) — worth revisiting in Phase 7 calibration.

### v3.2 — v3 skill + v2 derived (combo test)
- Val log_loss 0.6191, *worse* than v3 alone (0.6119). Brier worse, ECE between v2 and v3.
- best_iter=213 (vs v2's 642): early overfit signature → derived features add redundant signal on top of skill.
- v3's posterior implicitly captures fight frequency / recency; v2's layoff/activity columns duplicate it.
- **Kill. v3 stays current best.** Don't propagate v2 features into v3.1 / v4.


## Betting EV — would each model actually make money?

Reads [artifacts/metrics/bet_eval.json](../artifacts/metrics/bet_eval.json), produced by
[scripts/compare_models_bet_ev.py](../scripts/compare_models_bet_ev.py). Re-run that script after
training a new model version, then reload this notebook.

We simulate flat-$1 betting against the market on val 2023 fights. The decision rule: for each fight,
compute model expected return on each side; bet the side with EV ≥ `edge_threshold` (default 5%).

**Three market scenarios:**

| Scenario | Decimal odds source | Fee on winnings | What it represents |
|---|---|---:|---|
| **sportsbook (with vig)** | raw R_odds / B_odds | 0% | BestFightOdds reality — vig is in the price |
| **no-vig, no fee** | fair `1 / p_no_vig` | 0% | hypothetical: pure model vs market consensus |
| **kalshi-like** | fair `1 / p_no_vig` | 7% | what Kalshi / Polymarket roughly look like — the user's actual target |

**Metrics:**
- `roi_pct` — realized PnL per $1 staked, % (the headline number)
- `mean_ev_pct` — what the model *expected* per bet; compare to roi_pct to spot overconfidence
- `ci95` — bootstrap 95% CI for roi_pct (variance is huge with ~300 bets, take headlines with salt)
- `sharpe` — roi / std per bet
- `hit_rate` — fraction of bets that won (sanity check, not a goal)

In [ ]:
BET_EVAL_PATH = ROOT / 'artifacts' / 'metrics' / 'bet_eval.json'

if not BET_EVAL_PATH.exists():
    print(f'⚠ {BET_EVAL_PATH.name} not found. Run:')
    print('   PYTHONPATH=src .conda/bin/python scripts/compare_models_bet_ev.py')
    bet_eval = None
else:
    with open(BET_EVAL_PATH) as f:
        bet_eval = json.load(f)
    print(f"Loaded bet_eval.json (headline threshold = {bet_eval['headline_threshold']:.0%})")
    print(f"Threshold sweep points: {bet_eval['thresholds']}")
    print(f"Scenarios: {list(bet_eval['scenarios'].keys())}")

### Headline ROI by scenario @ edge threshold = 5%

One row per model per scenario. Sorted within each block by ROI (best first). The bootstrap CI columns tell you
whether the ROI is statistically meaningful — at ~300 bets the noise is large enough that almost every CI
spans zero. Treat headlines as directional, not definitive.

In [ ]:
if bet_eval is not None:
    for scenario_label, scenario in bet_eval['scenarios'].items():
        df = pd.DataFrame(scenario['headline']).sort_values('roi_pct', ascending=False)
        print(f'\n=== {scenario_label} (fee={scenario["fee_rate"]:.0%}, '
              f'no_vig={scenario["use_no_vig"]}) ===')
        cols = ['model', 'n_bets', 'roi_pct', 'ci95_low', 'ci95_high',
                'mean_ev_pct', 'sharpe', 'hit_rate']
        out = df[cols].copy()
        out['model'] = out['model'].str.replace('_catboost', '').str.replace('catboost_', '')
        display(out.style.format({
            'roi_pct': '{:+.2f}', 'ci95_low': '{:+.2f}', 'ci95_high': '{:+.2f}',
            'mean_ev_pct': '{:+.2f}', 'sharpe': '{:+.3f}', 'hit_rate': '{:.3f}',
        }).background_gradient(subset=['roi_pct'], cmap='RdYlGn'))

### Visual: ROI across models and scenarios

Grouped bar chart so you can scan which model wins in which market. The error bars are the bootstrap 95% CI for
ROI — when they straddle zero, we can't statistically distinguish that model from break-even on this val window.
The dashed grey line at 0% is the break-even line. Anything above is profitable in expectation; below is losing.

In [ ]:
if bet_eval is not None:
    # Long-form table for plotting.
    rows = []
    for scenario_label, scenario in bet_eval['scenarios'].items():
        for h in scenario['headline']:
            rows.append({
                'scenario': scenario_label,
                'model': h['model'].replace('_catboost', '').replace('catboost_', ''),
                'roi_pct': h['roi_pct'],
                'ci_low': h['ci95_low'],
                'ci_high': h['ci95_high'],
            })
    plot_df = pd.DataFrame(rows)
    # Order models by Kalshi-like ROI (the user's actual target).
    kalshi = plot_df[plot_df['scenario'].str.startswith('kalshi')].sort_values('roi_pct', ascending=False)
    model_order = kalshi['model'].tolist()
    scenario_order = ['sportsbook (with vig)', 'no-vig, no fee', 'kalshi-like (no vig, 7% fee)']
    colors = {'sportsbook (with vig)': '#d62728',
              'no-vig, no fee': '#2ca02c',
              'kalshi-like (no vig, 7% fee)': '#1f77b4'}

    fig, ax = plt.subplots(figsize=(12, 6))
    n_models = len(model_order)
    bar_width = 0.28
    x = np.arange(n_models)

    for i, scenario in enumerate(scenario_order):
        sub = plot_df[plot_df['scenario'] == scenario].set_index('model').loc[model_order]
        # Asymmetric error bars: distance from roi_pct to each CI bound.
        err = np.vstack([sub['roi_pct'] - sub['ci_low'], sub['ci_high'] - sub['roi_pct']])
        ax.bar(x + (i - 1) * bar_width, sub['roi_pct'], bar_width,
               label=scenario, color=colors[scenario],
               yerr=err, error_kw={'alpha': 0.4, 'capsize': 3})

    ax.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(model_order, rotation=30, ha='right')
    ax.set_ylabel('ROI %  per $1 bet')
    ax.set_title('Betting ROI by model and market — val 2023 fights, edge threshold ≥ 5%')
    ax.legend(loc='lower left', framealpha=0.9)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

### Threshold sweep — does the model's edge hold up at stricter thresholds?

For each scenario, plot ROI vs edge threshold for every model. A trustworthy bettor's ROI should *rise* with
stricter threshold (high-edge signals are real). If ROI is flat or *falls* as threshold rises, the model is
overconfident in its disagreements with the market — the "fake edge" pathology.

The Kalshi-like panel is the one to watch for the user's stated goal.

In [ ]:
if bet_eval is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    scenario_order = ['sportsbook (with vig)', 'no-vig, no fee', 'kalshi-like (no vig, 7% fee)']
    cmap = plt.cm.tab10

    for ax, scenario in zip(axes, scenario_order):
        sweep = pd.DataFrame(bet_eval['scenarios'][scenario]['sweep'])
        sweep['model'] = sweep['model'].str.replace('_catboost', '').str.replace('catboost_', '')
        # Order by Kalshi-like headline ROI for consistent coloring across panels.
        models_in_order = pd.DataFrame(bet_eval['scenarios']['kalshi-like (no vig, 7% fee)']['headline'])
        models_in_order['model'] = models_in_order['model'].str.replace('_catboost', '').str.replace('catboost_', '')
        models_in_order = models_in_order.sort_values('roi_pct', ascending=False)['model'].tolist()

        for i, m in enumerate(models_in_order):
            sub = sweep[sweep['model'] == m].sort_values('edge_threshold')
            ax.plot(sub['edge_threshold'] * 100, sub['roi_pct'], 'o-',
                    color=cmap(i % 10), label=m, linewidth=2 if 'v3_3' in m else 1.2,
                    alpha=0.9 if 'v3_3' in m else 0.7)

        ax.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.6)
        ax.set_xlabel('Edge threshold (%)')
        ax.set_title(scenario, fontsize=10)
        ax.grid(alpha=0.3)

    axes[0].set_ylabel('ROI %  per $1 bet')
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, framealpha=0.9)
    plt.tight_layout(); plt.show()

### Overconfidence diagnostic — model EV vs realized ROI

The dominant pathology in the current models: every one of them claims a ~40% EV per bet but realizes ~−5%
in practice. Plotting `mean_ev_pct` against `roi_pct` makes the gap visible — a well-calibrated model should
sit near the diagonal `roi == mean_ev`. **All current models sit far below the diagonal** — Phase 7
calibration is supposed to close this gap.

In [ ]:
if bet_eval is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    cmap = plt.cm.tab10
    markers = {'sportsbook (with vig)': 'o', 'no-vig, no fee': 's', 'kalshi-like (no vig, 7% fee)': '^'}

    for scenario_label, scenario in bet_eval['scenarios'].items():
        df = pd.DataFrame(scenario['headline'])
        df['model'] = df['model'].str.replace('_catboost', '').str.replace('catboost_', '')
        ax.scatter(df['mean_ev_pct'], df['roi_pct'],
                   marker=markers[scenario_label], s=110,
                   label=scenario_label, alpha=0.75, edgecolor='black', linewidth=0.5)

        if scenario_label.startswith('kalshi'):
            for _, row in df.iterrows():
                ax.annotate(row['model'], (row['mean_ev_pct'], row['roi_pct']),
                            fontsize=7, xytext=(4, 4), textcoords='offset points', alpha=0.8)

    lim_lo = min(0, plot_df['roi_pct'].min() - 2)
    lim_hi = max(50, 5)
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', alpha=0.4, label='perfect calibration (roi = ev)')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
    ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
    ax.set_xlabel('Model expected EV per bet (%)')
    ax.set_ylabel('Realized ROI per bet (%)')
    ax.set_title('Overconfidence gap: model expectation vs reality\n(per scenario, edge ≥ 5%)')
    ax.legend(loc='upper left', fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print('Diagonal = perfectly calibrated. Below diagonal = overconfident in disagreements.')
    print('Closing this gap is Phase 7\'s job.')

## Kelly bankroll simulation

Flat-stake ROI answers "per dollar bet, how much did I make on average?". Kelly answers a harder question:
**if I let bets compound at the mathematically-optimal stake size, where did my bankroll end up — and how bad
did the drawdowns get along the way?**

Each bet's stake is `bankroll × min(¼ × full_Kelly, 2%)`. The fraction (quarter-Kelly) plus the hard cap (2% of
bankroll per bet) are the PLAN.md §10.2 safety net. Starting bankroll = $1. Edge threshold = 3% (slightly looser
than flat-stake's 5% — Kelly handles smaller edges via smaller stakes).

Pay attention to **max drawdown**: PLAN.md §10.3 says "step down to 1/8 Kelly at −25% drawdown." Even our best
model on val brushes that line, which is a real-world risk signal.

In [ ]:
if bet_eval is not None:
    for scenario_label, scenario in bet_eval['scenarios'].items():
        if 'kelly' not in scenario:
            print(f"⚠ {scenario_label}: no kelly data (re-run scripts/compare_models_bet_ev.py)")
            continue
        rows = []
        for r in scenario['kelly']['rows']:
            rows.append({
                'model': r['model'].replace('_catboost', '').replace('catboost_', ''),
                'n_bets': r['n_bets'],
                'final_bankroll': r['final_bankroll'],
                'total_return_pct': r['total_return_pct'],
                'max_drawdown_pct': r['max_drawdown_pct'],
            })
        df = pd.DataFrame(rows).sort_values('total_return_pct', ascending=False)
        cfg = scenario['kelly']
        print(f"\n=== {scenario_label}  ({cfg['kelly_fraction']:.0%}-Kelly, "
              f"{cfg['max_bet_fraction']:.0%} cap, "
              f"{cfg['edge_threshold']:.0%} edge), starting bankroll $1 ===")
        display(df.style.format({
            'final_bankroll': '${:.3f}', 'total_return_pct': '{:+.2f}%',
            'max_drawdown_pct': '{:.2f}%',
        }).background_gradient(subset=['total_return_pct'], cmap='RdYlGn'))

### Bankroll trajectories — Kalshi-like scenario

The trajectory plot shows how each model's $1 bankroll evolves across the sequence of bets in val 2023. Models
above the dashed break-even line ended profitable; below ended in the red. Steep dips are drawdowns —
PLAN.md's §10.3 step-down rule triggers at −25%, marked.

In [ ]:
if bet_eval is not None:
    scenario_label = 'kalshi-like (no vig, 7% fee)'
    scenario = bet_eval['scenarios'].get(scenario_label, {})
    rows = scenario.get('kelly', {}).get('rows', [])
    # Order by final bankroll so the legend reads top-down by performance.
    rows = sorted(rows, key=lambda r: r['final_bankroll'], reverse=True)

    fig, ax = plt.subplots(figsize=(11, 6))
    cmap = plt.cm.tab20
    for i, r in enumerate(rows):
        traj = r.get('trajectory', [])
        name = r['model'].replace('_catboost', '').replace('catboost_', '')
        is_champion = 'v3_full2000' in r['model']
        is_v7 = 'v7' in r['model'] and 'v7_1' not in r['model']
        lw = 2.5 if is_champion else (2.0 if is_v7 else 1.0)
        alpha = 1.0 if (is_champion or is_v7) else 0.55
        ax.plot(traj, color=cmap(i % 20), label=f"{name} → ${r['final_bankroll']:.2f}",
                linewidth=lw, alpha=alpha)

    ax.axhline(1.0, color='black', linestyle='--', alpha=0.5, label='break-even')
    ax.axhline(0.75, color='red', linestyle=':', alpha=0.4, label='−25% step-down line')
    ax.set_xlabel('Bet # (chronological order)')
    ax.set_ylabel('Bankroll (started at $1)')
    ax.set_title(f"Kelly bankroll trajectory — {scenario_label}")
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, framealpha=0.9)
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print("Bold lines: v3_full2000 (champion) and v7 (conservative deploy alt).")
    print("Dotted red line at $0.75 = −25% drawdown → PLAN.md §10.3 says step down to 1/8 Kelly.")